# ============================================================
# [Data Integration] Dental Offices in Berlin
# ============================================================

# Install dependencies (if not already installed)
# %pip install osmnx geopandas pandas

# ============================================================
# 0. Imports
# ============================================================

In [559]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
from pathlib import Path
import numpy as np


# ============================================================
# 1. Data Extraction & Initial Inspection
# ============================================================


In [560]:
# 1.1 OSM Settings
ox.settings.use_cache = True
ox.settings.log_console = True

# 1.2 Fetch Dental Offices from OpenStreetMap
tags = {"amenity": "dentist"}

dental_offices_osm = ox.features.features_from_place(
    "Berlin, Germany",
    tags=tags
)

print(f"Number of dental office entries fetched: {len(dental_offices_osm)}")
print(dental_offices_osm.head(3).T.to_string())
print(dental_offices_osm['healthcare:speciality'].value_counts())

Number of dental office entries fetched: 798
element                                                  node                                                                                                                                                   
id                                                  304183504                                                                       313539258                                                          325161442
geometry                         POINT (13.612096 52.5114112)                                                   POINT (13.3553052 52.5488382)                                      POINT (13.1804772 52.5088434)
addr:city                                              Berlin                                                                          Berlin                                                                NaN
addr:country                                               DE                                                          

/Users/alex/anaconda3/envs/ds/lib/python3.10/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/alex/anaconda3/envs/ds/lib/python3.10/site-packages/shapely/set_operations.py:451: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


In [561]:
# Save raw data for reproducibility
dental_offices_osm.to_csv("../sources/raw_osm_dental_offices_v_01_19_2026.csv", index=False)
dental_offices_osm.to_file("../sources/raw_osm_dental_offices_v_01_19_2026.geojson", driver="GeoJSON")

# Inspect dataset
dental_offices_osm.info()
print(dental_offices_osm.columns.tolist())

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Columns: 104 entries, geometry to type
dtypes: geometry(1), object(103)
memory usage: 689.8+ KB
['geometry', 'addr:city', 'addr:country', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'amenity', 'health_facility:type', 'health_specialty:dentistry', 'healthcare', 'medical_system:western', 'office', 'operator', 'description', 'level', 'name', 'opening_hours', 'wheelchair', 'phone', 'toilets:wheelchair', 'website', 'contact:website', 'contact:email', 'contact:fax', 'contact:phone', 'check_date:opening_hours', 'check_date', 'email', 'fax', 'healthcare:speciality', 'air_conditioning', 'internet_access', 'internet_access:fee', 'payment:bank_transfer', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'payment:paypal', 'source', 'toilets', 'entrance', 'opening_hours:signed', 'wheelchair:description', 'emergency', 'name:de', 'name:en', 'payment:cont

# ============================================================
# 2. Data Selection & Column Standardization
# ============================================================

In [ ]:
# Select key columns relevant for dental officesс
columns = [
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "level",
    "opening_hours",
    "check_date",
    "healthcare:speciality",
    "wheelchair",
    "wheelchair:description",
    "phone",
    "email",
    "website",
    "geometry",
    "health_facility:type",
    "health_specialty:oral_surgery",
    "health_specialty:orthodontics",
    "health_specialty:periodontology"
]

# Filter the dataset to keep only the selected columns
dental_offices = dental_offices_osm[[c for c in columns if c in dental_offices_osm.columns]].copy()

# ============================================================
# Next Processing Steps (Roadmap for future PRs / scripts)
# ============================================================
# 1. Name normalization
#    - Standardize names such as "Zahnarztpraxis", "Drs.", and other practice naming conventions.
# 2. Address cleaning
#    - Format street names and house numbers to a consistent structure.
# 3. Category mapping
#    - Map specialization fields into a controlled vocabulary for consistency.
# 4. Deduplication logic
#    - Detect and handle overlaps between OSM entries and official city registries.


# ============================================================
# 3. Speciality Mapping
# ============================================================

In [ ]:
# Mapping of OSM health specialty columns to standardized speciality names
# Only consider values marked as "yes" or "main"
special_cols = {
    "health_specialty:oral_surgery": "oral_surgery",
    "health_specialty:orthodontics": "orthodontics",
    "health_specialty:periodontology": "periodontology"
}

def compute_speciality(row):
    """
    Determine standardized speciality for a dental office.
    
    Priority:
    1. Use 'healthcare:speciality' if non-empty
    2. Check boolean-style specialty columns ("yes" or "main")
    3. Default to "Unknown"
    """
    val = row.get("healthcare:speciality")
    if pd.notna(val) and str(val).strip() != "":
        return str(val).strip()
    
    for col, name in special_cols.items():
        cell = row.get(col)
        if pd.notna(cell) and str(cell).lower() in ["yes", "main"]:
            return name
    
    return "Unknown"

# Apply speciality mapping
dental_offices["speciality"] = dental_offices.apply(compute_speciality, axis=1)

# Drop original speciality columns to avoid redundancy
dental_offices.drop(
    columns=["healthcare:speciality"] + list(special_cols.keys()), inplace=True
)

dental_offices.head()

name     addr:street addr:housenumber  \
element id                                                                
node    304183504                  NaN  Hönower Straße               75   
        313539258  Zahnzentrum Wedding    Müllerstraße              34a   
        325161442             A. Nejad             NaN              NaN   
        345236220    Dr. Beate Lengert  Kurfürstendamm              218   
        391394177      Serpil Hartfiel  Kollwitzstraße               77   

                  addr:postcode addr:city level  \
element id                                        
node    304183504         12623    Berlin   NaN   
        313539258         13353    Berlin     2   
        325161442           NaN       NaN   NaN   
        345236220         10719    Berlin   NaN   
        391394177         10435    Berlin   NaN   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   
        345236220                                                NaN   
        391394177  Mo,Tu,Th 08:00-19:00; We 18:00-18:00; Fr 08:00...   

                  check_date wheelchair wheelchair:description  \
element id                                                       
node    304183504        NaN        NaN                    NaN   
        313539258        NaN        yes                    NaN   
        325161442        NaN        yes                    NaN   
        345236220        NaN        NaN                    NaN   
        391394177        NaN         no                    NaN   

                              phone email                          website  \
element id                                                                   
node    304183504               NaN   NaN                              NaN   
        313539258               NaN   NaN                              NaN   
        325161442  +49 30 361 91 06   NaN                              NaN   
        345236220               NaN   NaN  http://www.dr-beate-lengert.de/   
        391394177               NaN   NaN                              NaN   

                                    geometry health_facility:type speciality  
element id                                                                    
node    304183504   POINT (13.6121 52.51141)               office    Unknown  
        313539258  POINT (13.35531 52.54884)                  NaN    Unknown  
        325161442  POINT (13.18048 52.50884)                  NaN    Unknown  
        345236220  POINT (13.32814 52.50272)                  NaN    Unknown  
        391394177  POINT (13.41899 52.53755)                  NaN    Unknown

# ============================================================
# 4. Geometry Processing
# ============================================================

In [ ]:

# Ensure point geometry
dental_offices["geometry"] = dental_offices["geometry"].apply(
    lambda g: g if g.geom_type == "Point" else g.representative_point()
)

# Extract latitude and longitude
dental_offices["latitude"] = dental_offices.geometry.y
dental_offices["longitude"] = dental_offices.geometry.x

# Standardize address columns
dental_offices = dental_offices.rename(columns={
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:city": "city"
})
dental_offices.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   name                    767 non-null    object  
 1   street                  582 non-null    object  
 2   housenumber             582 non-null    object  
 3   postcode                535 non-null    object  
 4   city                    527 non-null    object  
 5   level                   90 non-null     object  
 6   opening_hours           602 non-null    object  
 7   check_date              140 non-null    object  
 8   wheelchair              286 non-null    object  
 9   wheelchair:description  8 non-null      object  
 10  phone                   289 non-null    object  
 11  email                   88 non-null     object  
 12  website                 306 non-null    object  
 13  geometry                798 non-null   

# ============================================================
# 5. Neighborhood Assignment via Spatial Join
# ============================================================

In [ ]:

# Load neighborhoods GeoJSON (Berlin LOR Ortsteile)
lor_path = Path('../../mapping/lor_ortsteile.geojson')
neighborhoods = gpd.read_file(lor_path).to_crs("EPSG:4326")

# Rename for consistency
neighborhoods = neighborhoods.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
})

# Create GeoDataFrame for dental offices
dental_gdf = gpd.GeoDataFrame(dental_offices, geometry='geometry', crs='EPSG:4326')
print(f"✓ Created GeoDataFrame with {len(dental_gdf)} dental offices")


✓ Created GeoDataFrame with 798 dental offices


In [ ]:
# Spatial join to assign neighborhoods
df_with_districts = gpd.sjoin(
    dental_gdf,
    neighborhoods[["district", "neighborhood", "neighborhood_id", "geometry"]],
    how="left",
    predicate="within"
)

# Drop unnecessary columns from spatial join
df_final = df_with_districts.drop(columns=["geometry", "index_right"])
df_final.head(3)

name          street housenumber postcode  \
element id                                                                    
node    304183504                  NaN  Hönower Straße          75    12623   
        313539258  Zahnzentrum Wedding    Müllerstraße         34a    13353   
        325161442             A. Nejad             NaN         NaN      NaN   

                     city level  \
element id                        
node    304183504  Berlin   NaN   
        313539258  Berlin     2   
        325161442     NaN   NaN   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   

                  check_date wheelchair wheelchair:description  \
element id                                                       
node    304183504        NaN        NaN                    NaN   
        313539258        NaN        yes                    NaN   
        325161442        NaN        yes                    NaN   

                              phone email website health_facility:type  \
element id                                                               
node    304183504               NaN   NaN     NaN               office   
        313539258               NaN   NaN     NaN                  NaN   
        325161442  +49 30 361 91 06   NaN     NaN                  NaN   

                  speciality   latitude  longitude             district  \
element id                                                                
node    304183504    Unknown  52.511411  13.612096  Marzahn-Hellersdorf   
        313539258    Unknown  52.548838  13.355305                Mitte   
        325161442    Unknown  52.508843  13.180477              Spandau   

                   neighborhood neighborhood_id  
element id                                       
node    304183504     Mahlsdorf            1004  
        313539258       Wedding            0105  
        325161442  Wilhelmstadt            0509

# ============================================================
# 6. District ID Mapping
# ============================================================

In [ ]:
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

df_final['district_id'] = df_final['district'].map(district_mapping).astype(str)

# Check for unmapped districts
unmapped = df_final[~df_final['district'].isin(district_mapping.keys())]['district'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts found:", unmapped)

# ============================================================
# 7. Dental Office ID & Index Cleanup
# ============================================================

In [ ]:
df_final = df_final.reset_index()
df_final = df_final.drop(columns=["element"]).rename(columns={"id": "store_id"})
df_final["store_id"] = df_final["store_id"].astype("string")
print(df_final.shape[0], "total dental office records after processing.")
print(df_final["store_id"].nunique(), "unique dental offices IDs assigned.")
print(df_final.shape[0], "total dental office records after processing.")
print(df_final["store_id"].nunique(), "unique dental offices IDs assigned.")
df_final.head(2)

798 total dental office records after processing.
798 unique store IDs assigned.
798 total dental office records after processing.
798 unique store IDs assigned.


,store_id,name,street,housenumber,postcode,city,level,opening_hours,check_date,wheelchair,...,email,website,health_facility:type,speciality,latitude,longitude,district,neighborhood,neighborhood_id,district_id
0,304183504,NaN,Hönower Straße,75,12623,Berlin,NaN,NaN,NaN,NaN,...,NaN,NaN,office,Unknown,52.511411,13.612096,Marzahn-Hellersdorf,Mahlsdorf,1004,11010010
1,313539258,Zahnzentrum Wedding,Müllerstraße,34a,13353,Berlin,2,Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...,NaN,yes,...,NaN,NaN,NaN,Unknown,52.548838,13.355305,Mitte,Wedding,0105,11001001


# ============================================================
# 8. Level / Floor Mapping
# ============================================================

In [ ]:
# Mapping DEU, UK und US
LEVEL_MAP_DEU = {
    "-1": "UG", "0": "EG", "1": "1.OG", "2": "2.OG", "3": "3.OG",
    "4": "4.OG", "5": "5.OG", "6": "6.OG", "7": "7.OG", "8": "8.OG", "9": "9.OG", "10": "10.OG"
}

LEVEL_MAP_UK = {
    "-1": "Basement", "0": "Ground Floor", "1": "First Floor", "2": "Second Floor",
    "3": "Third Floor", "4": "Fourth Floor", "5": "Fifth Floor", "6": "Sixth Floor"
}

LEVEL_MAP_US = {
    "-1": "Basement", "0": "First Floor", "1": "Second Floor", "2": "Third Floor",
    "3": "Fourth Floor", "4": "Fifth Floor", "5": "Sixth Floor", "6": "Seventh Floor"
}
# Select mapping (DEU / UK / US)
LEVEL_MAP = LEVEL_MAP_DEU  # Change to LEVEL_MAP_UK or LEVEL_MAP_US as needed

def format_level(level):
    """Convert numeric level to standardized floor label."""
    if pd.isna(level):
        return ""
    return LEVEL_MAP.get(str(level).strip(), level)




# ============================================================
# 9. Address Formatting
# ============================================================

In [ ]:
def format_address(row):
    """
    Construct full address string with optional floor and neighborhood.
    If all core address fields are missing, return 'Unknown'.
    """
    # If all core fields are missing
    if all(pd.isna(row[col]) for col in ['street', 'housenumber', 'level', 'postcode', 'city']):
        return "Unknown"
    
    # Prepare components
    street = row['street'] if pd.notna(row['street']) else ""
    housenumber = row['housenumber'] if pd.notna(row['housenumber']) else ""
    postcode = row['postcode'] if pd.notna(row['postcode']) else ""
    city = row['city'] if pd.notna(row['city']) else ""
    neighborhood = row['neighborhood'] if pd.notna(row['neighborhood']) else ""
    
    # Floor formatting
    level_str = format_level(row['level'])
    
    # Construct address parts
    parts = [f"{street} {housenumber}".strip()]
    if level_str:
        parts.append(level_str)
    city_line = f"{postcode} {city}".strip()
    if neighborhood:
        city_line += f"-{neighborhood}"
    parts.append(city_line)
    
    return ", ".join([p for p in parts if p])

# Apply formatted address
df_final["address"] = df_final.apply(format_address, axis=1)

# Drop individual address columns to clean up
df_final.drop(columns=["street", "housenumber", "postcode", "city", "level"], inplace=True)


# ============================================================
# 10. Data Overview / Quality Check (Columns, Types, Missing Values)
# ============================================================

In [ ]:
# Missing Values
missing = df_final.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_final) * 100).round(1)
print('Data types:')
display(df_final.dtypes)
print(f'Dental Offices Data: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}')
print("Missing Values Summary:")
pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})

Data types:


store_id                  string[python]
name                              object
opening_hours                     object
check_date                        object
wheelchair                        object
wheelchair:description            object
phone                             object
email                             object
website                           object
health_facility:type              object
speciality                        object
latitude                         float64
longitude                        float64
district                          object
neighborhood                      object
neighborhood_id                   object
district_id                       object
address                           object
dtype: object

Dental Offices Data: Rows: 798, Columns: 18
Missing Values Summary:


,missing_count,missing_pct
wheelchair:description,790,99.0
health_facility:type,775,97.1
email,710,89.0
check_date,658,82.5
wheelchair,512,64.2
phone,509,63.8
website,492,61.7
opening_hours,196,24.6
name,31,3.9
district,0,0.0
